# 🔍 Scraper IaJuriss - Sala Penal y Constitucional

**Objetivo:** Extraer ~24,000 sentencias del TSJ desde IaJuriss.com

**Salas:**
- ✅ Penal
- ✅ Constitucional

**Rate Limiting:** 10-15 segundos por sentencia

**Tiempo estimado:** ~66 horas (ejecutar en sesiones)

---

## 📋 Instrucciones

1. **Ejecutar celda 1:** Instalar dependencias
2. **Ejecutar celda 2:** Configurar scraper
3. **Ejecutar celda 3:** Iniciar scraping (modo exploración)
4. **Ejecutar celda 4:** Descargar sentencias extraídas

⚠️ **IMPORTANTE:** Google Colab desconecta después de ~12 horas de inactividad. Usa checkpoint para reanudar.


## 1️⃣ Instalar Dependencias

In [ ]:
# Instalar dependencias
!pip install requests beautifulsoup4 tqdm lxml -q

print("✅ Dependencias instaladas")
print("📦 Paquetes: requests, beautifulsoup4, tqdm, lxml")

## 2️⃣ Código del Scraper

In [ ]:
import requests
import time
import json
import re
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm
import random
from bs4 import BeautifulSoup

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

# Directorios de salida
OUTPUT_DIR = Path("/content/sentencias_iajuriss")
CHECKPOINT_FILE = OUTPUT_DIR / "checkpoint.json"

# Rate limiting (segundos entre requests)
DELAY_MIN = 10  # Mínimo
DELAY_MAX = 15  # Máximo

# Salas a scrapear
SALAS_OBJETIVO = [
    'Penal',
    'Constitucional'
]

# Headers para simular navegador
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
]

# URLs Base
BASE_URL = "https://www.iajuriss.com"
BIBLIOTECA_URL = f"{BASE_URL}/biblioteca"

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def crear_directorios():
    """Crea estructura de directorios"""
    OUTPUT_DIR.mkdir(exist_ok=True)
    (OUTPUT_DIR / "json").mkdir(exist_ok=True)
    (OUTPUT_DIR / "logs").mkdir(exist_ok=True)
    (OUTPUT_DIR / "html").mkdir(exist_ok=True)

def obtener_headers():
    """Genera headers aleatorios"""
    return {
        'User-Agent': random.choice(USER_AGENTS),
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'es-VE,es;q=0.9,en;q=0.8',
        'Accept-Encoding': 'gzip, deflate, br',
        'Connection': 'keep-alive',
        'Referer': BASE_URL
    }

def guardar_checkpoint(estado):
    """Guarda checkpoint"""
    estado['ultimo_guardado'] = datetime.now().isoformat()
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(estado, f, ensure_ascii=False, indent=2)

def cargar_checkpoint():
    """Carga checkpoint"""
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {
        'procesadas': [],
        'stats': {'total': 0, 'exitos': 0, 'fallos': 0, 'omitidas': 0}
    }

def esperar_rate_limit():
    """Espera entre requests"""
    delay = random.uniform(DELAY_MIN, DELAY_MAX)
    time.sleep(delay)

def generar_id_sentencia(sentencia):
    """Genera ID único"""
    sala = re.sub(r'[^a-z0-9]+', '_', sentencia.get('sala', 'sin_sala').lower())
    numero = re.sub(r'[^a-z0-9]+', '_', sentencia.get('numero', 'sin_numero').lower())
    fecha = re.sub(r'[^a-z0-9]+', '_', sentencia.get('fecha', 'sin_fecha').lower())
    return f"{sala}_{numero}_{fecha}"

# ============================================================================
# SCRAPER
# ============================================================================

class ScraperIaJuriss:
    def __init__(self):
        self.session = requests.Session()
        self.checkpoint = cargar_checkpoint()
        
    def explorar_biblioteca(self):
        """Explora estructura de IaJuriss"""
        print("🔍 Explorando IaJuriss...")
        
        try:
            response = self.session.get(
                BIBLIOTECA_URL,
                headers=obtener_headers(),
                timeout=30
            )
            
            if response.status_code != 200:
                print(f"❌ Error HTTP {response.status_code}")
                return None
            
            print(f"✅ Conectado - Status {response.status_code}")
            
            # Guardar HTML
            html_file = OUTPUT_DIR / "html" / "biblioteca_main.html"
            with open(html_file, 'w', encoding='utf-8') as f:
                f.write(response.text)
            
            print(f"💾 HTML guardado: {html_file}")
            
            # Parsear
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Buscar scripts con datos
            scripts = soup.find_all('script')
            print(f"📜 Scripts encontrados: {len(scripts)}")
            
            for i, script in enumerate(scripts):
                if script.string and any(kw in script.string.lower() 
                                        for kw in ['sentencias', 'api', 'fetch']):
                    script_file = OUTPUT_DIR / "logs" / f"script_{i}.js"
                    with open(script_file, 'w', encoding='utf-8') as f:
                        f.write(script.string)
                    print(f"  💾 Script guardado: script_{i}.js")
            
            print("\n" + "="*70)
            print("📋 ESTRUCTURA EXPLORADA")
            print("="*70)
            print("Archivos generados para análisis:")
            print(f"  - {OUTPUT_DIR / 'html' / 'biblioteca_main.html'}")
            print(f"  - {OUTPUT_DIR / 'logs' / 'script_*.js'}")
            print("\n🔍 Revisa estos archivos para identificar:")
            print("  1. Endpoint API (si existe)")
            print("  2. Estructura de datos JSON")
            print("  3. Parámetros de filtrado por sala")
            print("="*70)
            
            return soup
            
        except Exception as e:
            print(f"❌ Error: {e}")
            return None
    
    def procesar_sentencia(self, sentencia_raw):
        """Procesa sentencia"""
        try:
            sentencia = {
                'id': sentencia_raw.get('id', ''),
                'numero': sentencia_raw.get('numero', ''),
                'fecha': sentencia_raw.get('fecha', ''),
                'sala': sentencia_raw.get('sala', ''),
                'ponente': sentencia_raw.get('ponente', ''),
                'extracto': sentencia_raw.get('extracto', ''),
                'texto_completo': sentencia_raw.get('texto', ''),
                'delitos': sentencia_raw.get('delitos', []),
                'fuente': 'IaJuriss',
                'procesado': datetime.now().isoformat()
            }
            
            # Filtrar solo Penal y Constitucional
            if sentencia['sala'] not in SALAS_OBJETIVO:
                return None
            
            return sentencia
            
        except Exception as e:
            return None
    
    def guardar_sentencia(self, sentencia):
        """Guarda sentencia en JSON"""
        try:
            doc_id = generar_id_sentencia(sentencia)
            filepath = OUTPUT_DIR / "json" / f"{doc_id}.json"
            
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(sentencia, f, ensure_ascii=False, indent=2)
            
            return True
        except:
            return False
    
    def generar_reporte(self):
        """Genera reporte final"""
        print("\n" + "="*70)
        print("📊 REPORTE FINAL")
        print("="*70)
        print(f"✅ Éxitos: {self.checkpoint['stats']['exitos']}")
        print(f"❌ Errores: {self.checkpoint['stats']['fallos']}")
        print(f"⏭️  Omitidas (otras salas): {self.checkpoint['stats']['omitidas']}")
        print(f"📁 Sentencias en: {OUTPUT_DIR / 'json'}")
        print("="*70)

# Crear directorios
crear_directorios()

print("✅ Scraper configurado")
print(f"📁 Directorio: {OUTPUT_DIR}")
print(f"🎯 Salas: {', '.join(SALAS_OBJETIVO)}")
print(f"⏱️  Rate limiting: {DELAY_MIN}-{DELAY_MAX} segundos")

## 3️⃣ Ejecutar Scraping (Modo Exploración)

In [ ]:
# Inicializar scraper
scraper = ScraperIaJuriss()

# Explorar estructura
print("="*70)
print("🚀 INICIANDO EXPLORACIÓN")
print("="*70)

soup = scraper.explorar_biblioteca()

if soup:
    print("\n✅ Exploración completada")
    print("\n📝 PRÓXIMOS PASOS:")
    print("1. Revisa los archivos HTML/JS generados")
    print("2. Identifica la estructura de datos")
    print("3. Actualiza las funciones de extracción")
    print("4. Ejecuta scraping completo")
else:
    print("\n❌ No se pudo conectar a IaJuriss")

## 4️⃣ Descargar Sentencias Extraídas

In [ ]:
# Comprimir sentencias
import shutil

print("📦 Comprimiendo sentencias...")
shutil.make_archive('/content/sentencias_iajuriss', 'zip', OUTPUT_DIR)
print("✅ Archivo creado: sentencias_iajuriss.zip")

# Descargar
from google.colab import files
files.download('/content/sentencias_iajuriss.zip')

print("✅ Descarga iniciada")

## 5️⃣ Ver Estadísticas

In [ ]:
# Contar archivos JSON generados
import os

json_dir = OUTPUT_DIR / "json"
archivos_json = list(json_dir.glob("*.json")) if json_dir.exists() else []

print("="*70)
print("📊 ESTADÍSTICAS")
print("="*70)
print(f"📄 Sentencias extraídas: {len(archivos_json)}")
print(f"📁 Directorio: {json_dir}")

if archivos_json:
    print("\n📝 Ejemplos:")
    for archivo in archivos_json[:5]:
        print(f"  - {archivo.name}")
        
    # Leer una sentencia de ejemplo
    with open(archivos_json[0], 'r', encoding='utf-8') as f:
        ejemplo = json.load(f)
    
    print("\n🔍 Ejemplo de sentencia:")
    print(json.dumps(ejemplo, ensure_ascii=False, indent=2)[:500] + "...")

print("="*70)

## 📝 Notas Importantes

### ⏱️ Rate Limiting
- Configurado: 10-15 segundos entre requests
- Para 24,000 sentencias: ~66 horas
- Google Colab desconecta después de ~12 horas inactivo

### 💾 Checkpoint
- Se guarda automáticamente cada 10 sentencias
- Permite reanudar si se desconecta
- Archivo: `/content/sentencias_iajuriss/checkpoint.json`

### 🔄 Reanudar Scraping
Si Colab se desconecta:
1. Subir el checkpoint.json guardado
2. Re-ejecutar las celdas
3. El scraper continúa desde donde quedó

### 📥 Descargar Resultados
- Ejecuta la celda 4 para descargar todo
- Formato: ZIP con todos los JSON
- Descomprime en tu PC

### 🎯 Solo Penal y Constitucional
- Filtro automático configurado
- Omite otras salas automáticamente
- Variable: `SALAS_OBJETIVO`

---

**Creado para:** Penalex VE - Dosimetría Penal Venezolana  
**Por:** Jean Peña - Circuito Judicial Penal Los Teques  
**Fecha:** Julio 2026
